In [ ]:
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

import os
from dotenv import load_dotenv

load_dotenv()

# 1. Load pdf
documents = PyPDFLoader("./files/employee_handbook_v2.pdf").load()
print(f"Loaded documents: {len(documents)}")

# 2. Chunkerization

# 2.1 create splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len)

# 2.2 create chunks
docs = text_splitter.split_documents(documents)
print (f"Created chunks: {len(docs)}")

# 3. Create DB

persist_directory = "./chroma_db"
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

if os.path.exists(persist_directory):
    print("Already created.")
    vector_store = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
else:
    print("Not yet created. Generating Vector DB.")
    vector_store = Chroma.from_documents(docs, embeddings, persist_directory=persist_directory)
    
print("Vector DB created/loaded.")

retriever = vector_store.as_retriever(search_kwargs={"k": 2})

llm = init_chat_model("openai:gpt-5.4-mini", temperature=0)

user_query = input("How does the company's founding mission relate to the specific technology powering the ships, and what time must the weekly maintenance logs for that technology be submitted?")

similar_docs = retriever.invoke(user_query)

llm_context = ""


Loaded documents: 10
Created chunks: 15
Already created.
Vector DB created/loaded.
